In [1]:
# %% [1. RL GRIDWORLD ENVIRONMENT]
import numpy as np
import matplotlib.pyplot as plt

ROWS, COLS = 4, 4
START = (0, 0)
GOAL = (3, 3)
PIT = (1, 1)
WALL = (2, 2)

ACTIONS = ['U', 'D', 'L', 'R']
ARROWS = {'U': '↑', 'D': '↓', 'L': '←', 'R': '→'}
MOVES = {'U': (-1, 0), 'D': (1, 0), 'L': (0, -1), 'R': (0, 1)}

def step(state, action):
    if state in [GOAL, PIT]:
        return state, 0.0, True
    dr, dc = MOVES[action]
    nr, nc = state[0] + dr, state[1] + dc
    if nr < 0 or nr >= ROWS or nc < 0 or nc >= COLS or (nr, nc) == WALL:
        nr, nc = state
    next_state = (nr, nc)
    if next_state == GOAL:
        return next_state, 10.0, True
    elif next_state == PIT:
        return next_state, -10.0, True
    return next_state, -0.1, False

def show_policy(policy_grid, title):
    print(f"\n=== {title} ===")
    for r in range(ROWS):
        row_str = []
        for c in range(COLS):
            s = (r, c)
            if s == GOAL: row_str.append(' G ')
            elif s == PIT: row_str.append(' P ')
            elif s == WALL: row_str.append(' # ')
            else: row_str.append(f' {policy_grid[r][c]} ')
        print(' '.join(row_str))

print("States: 16 | Start: (0,0) | Goal (+10): (3,3) | Pit (-10): (1,1) | Wall: (2,2) | Step: -0.1")


States: 16 | Start: (0,0) | Goal (+10): (3,3) | Pit (-10): (1,1) | Wall: (2,2) | Step: -0.1


In [2]:
# %% [2A. MDP - VALUE ITERATION]
gamma = 0.9
V = np.zeros((ROWS, COLS))

for _ in range(200):
    V_new = np.copy(V)
    for r in range(ROWS):
        for c in range(COLS):
            s = (r, c)
            if s in [GOAL, PIT, WALL]:
                continue
            q_vals = []
            for a in ACTIONS:
                ns, rew, done = step(s, a)
                q_vals.append(rew + (0.0 if done else gamma * V[ns]))
            V_new[s] = max(q_vals)
    if np.max(np.abs(V_new - V)) < 1e-4:
        break
    V = V_new

mdp_policy = [['' for _ in range(COLS)] for _ in range(ROWS)]
for r in range(ROWS):
    for c in range(COLS):
        s = (r, c)
        if s in [GOAL, PIT, WALL]:
            continue
        q_vals = [step(s, a)[1] + (0.0 if step(s, a)[2] else gamma * V[step(s, a)[0]]) for a in ACTIONS]
        mdp_policy[r][c] = ARROWS[ACTIONS[int(np.argmax(q_vals))]]

print("MDP State-Value Grid V(s):\n", np.round(V, 2))
show_policy(mdp_policy, "MDP Optimal Policy")


MDP State-Value Grid V(s):
 [[ 5.5   6.22  7.02  7.91]
 [ 6.22  0.    7.91  8.9 ]
 [ 7.02  7.91  0.   10.  ]
 [ 7.91  8.9  10.    0.  ]]

=== MDP Optimal Policy ===
 ↓   →   ↓   ↓ 
 ↓   P   →   ↓ 
 ↓   ↓   #   ↓ 
 →   →   →   G 
